In [ ]:
'''
    Code to calculate OR across contexts using logistic regression as alternate to Pain et al. 2022
'''

import pandas as pd
import numpy as np
import random
import time

from options.options import Options
import util.util as util
import util.pre_process as pre
import util.bootstrap_tools as btool

from scipy.stats import norm

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import numpy as np

phen_label = 'MDD' # choose phenotype

def gen_results(pgs, phen, covars, pgs_res, phen_res):
    results = {}

    # Compute R²
    r2_results = btool.efficient_r2(pgs_res, phen_res)
    results['r2_obs'] = r2_results['r2']

    # Compute phenotype prevalence and proportion of individuals with high PGS
    results['prev'] = phen[None,:].mean(1)
    results['high_pgs'] = (pgs > emerge_cutoff_z).mean()

    # Compute odds ratios based on R² and prevalence
    results['or'] = btool.efficient_odds_ratio(results['r2_obs'], results['prev'])
    results['top_or'] = btool.effecient_r2_to_risk(results['r2_obs'], results['prev'], emerge_cutoff)['odds_ratio']
    results['top_quant_or'] = btool.effecient_r2_to_risk(results['r2_obs'], results['prev'], top_quant)['odds_ratio']

    # Prepare covariate matrices
    pgs_covars = np.hstack([pgs[:, None], covars])
    top_quant_pgs_covars = np.hstack([pgs[:, None] >= top_cutoff_z, covars])
    top_pgs_covars = np.hstack([pgs[:, None] >= emerge_cutoff_z, covars])

    # Perform logistic regression
    log_reg_results = btool.efficient_logistic_regression(pgs_covars, phen, ['weights', 'converged'])
    top_quant_log_reg_results = btool.efficient_logistic_regression(top_quant_pgs_covars, phen, ['weights', 'converged'])
    top_log_reg_results = btool.efficient_logistic_regression(top_pgs_covars, phen, ['weights', 'converged'])

    # Extract and exponentiate logistic regression coefficients
    results['or_log'] = np.exp(log_reg_results['weights'][:, 0])
    results['top_quant_or_log'] = np.exp(top_quant_log_reg_results['weights'][:, 0])
    results['top_or_log'] = np.exp(top_log_reg_results['weights'][:, 0])

    results['or_log_converged'] = log_reg_results['converged']
    results['top_quant_or_log_converged'] = top_quant_log_reg_results['converged']
    results['top_or_log_converged'] = top_log_reg_results['converged']

    return results

opt = Options()
opt.initialize()

top_quant = .25
top_cutoff_z = norm.ppf(1 - top_quant)

emerge_cutoff = opt.phen_params[phen_label]['emerge_cut']
emerge_cutoff_z = norm.ppf(1 - emerge_cutoff)

pgs_df = pd.read_csv(opt.prs_file, delim_whitespace=True, index_col='IID')
covars_df = pd.read_csv(opt.env_file, sep='\t', index_col='IID')

pc_cols = ['chip']+[df_col for df_col in covars_df.columns if df_col.startswith('PC')]
pcs_df = covars_df[pc_cols].iloc[:, :11]
covars_df = covars_df[opt.covars]

phen = pgs_df[opt.phen_params[phen_label]['phen']]
phen = phen.dropna()
pgs = pgs_df[opt.phen_params[phen_label]['pgs']].loc[phen.index]
pgs = (pgs-pgs.mean())/pgs.std()
if opt.phen_params[phen_label]['pgs_flip']:
    pgs = -1*pgs
covars_df = covars_df.loc[phen.index]

covars_imputed_df = pre.impute_covars(covars_df)
covars_imputed_df, _, _ = pre.standardize_covars(covars_imputed_df)
covars = pd.concat([covars_imputed_df, pre.standardize_covars(pcs_df)[0]], axis=1)
covars = covars.loc[pgs.index]

phen_res, _ = pre.lin_regress_out(covars, phen)
pgs_res, _ = pre.lin_regress_out(covars, pgs)

bin_defs = opt.bin_defs
bounds, labels, label_map = util.create_bins_and_indices(covars_df, bin_defs)

util.create_folder(f'results/{phen_label}')

print('\n')
print(f'Total: {len(phen)}')
for sex_label in ['Female', 'Male']:
    n_sex = len(phen.loc[label_map['Sex: '+sex_label]])
    n_cases = phen.loc[label_map['Sex: '+sex_label]].sum()
    print(f'{sex_label} Cases: {n_cases}/{n_sex}')

for k in label_map.keys():
    print(len(label_map[k]), ': ', k,)

rng = np.random.default_rng(np.random.PCG64(42))

In [ ]:
import concurrent.futures

# Initialize arrays
n = len(labels)
vars = ['n', 'high_pgs', 'prev', 
        'or_log', 'top_quant_or_log', 'top_or_log', 
        'or_log_converged', 'top_quant_or_log_converged', 'top_or_log_converged', 
        'or', 'top_quant_or', 'top_or']

results = {var: np.full((n, n), np.nan) for var in vars+['n']}

## Intersect analysis
for i in range(n):
    for j in range(i):
        print(i,j)
        
        idx_org = np.intersect1d(label_map[labels[i]], label_map[labels[j]])
        if len(idx_org) == 0:
            continue
        
        results['n'][i, j] = results['n'][j, i] = len(idx_org)

        pgs_ = pgs.loc[idx_org].values
        pgs_ = (pgs_-pgs_.mean())/pgs_.std() # to standardize in context
        
        phen_ = phen.loc[idx_org].values
        covars_ = covars.loc[idx_org].values
        x = pgs_res.loc[idx_org].values
        y = phen_res.loc[idx_org].values

        results_ = gen_results(pgs_, phen_, covars_, x, y)
        for key in results_:
            if key not in vars:
                continue
            results[key][i, j] = results[key][j, i] = results_[key].item()
np.save(f'results/{phen_label}/gt_comparisons_v2.npy', results)